# 06 — Hyperparameter Search: RandomizedSearchCV vs Optuna (TPE)

The model in notebook 02 uses hand-picked hyperparameters (`max_depth=6, learning_rate=0.05`). Here we search for better ones — and compare **two search methods head-to-head** on the same search space, CV protocol, and trial budget:

- **Cost**: wall-clock time + trials-to-95%-of-final-best (sample efficiency)
- **Performance**: final CV PR-AUC, then held-out test PR-AUC / F-β=2 / cost vs. the manual baseline

Outputs:
- `models/fraud_xgb_tuned.pkl`
- `data_export.json` → `hyperparam_search`

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
from optuna.samplers import TPESampler
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import average_precision_score, fbeta_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from xgboost import XGBClassifier

optuna.logging.set_verbosity(optuna.logging.WARNING)

DATA_DIR  = Path("../../..") / "data" / "ieee-fraud-detection"
MODEL_DIR = Path("../../..") / "models"
DEMO_DIR  = Path("..") / "demo"
EXPORT_PATH = DEMO_DIR / "data_export.json"

BASELINE_PARAMS = {
    "max_depth": 6, "learning_rate": 0.05, "min_child_weight": 1,
    "subsample": 1.0, "colsample_bytree": 1.0, "gamma": 0.0,
    "reg_alpha": 0.0, "reg_lambda": 1.0,
}
SEARCH_SPACE = {
    "max_depth": (3, 10), "learning_rate": (0.01, 0.3, "log"),
    "min_child_weight": (1, 10), "subsample": (0.5, 1.0),
    "colsample_bytree": (0.5, 1.0), "gamma": (0.0, 5.0),
    "reg_alpha": (1e-3, 10.0, "log"), "reg_lambda": (1e-3, 10.0, "log"),
}
N_TRIALS = 25
CV_FOLDS = 3
SEARCH_SUBSAMPLE = 60_000
SEARCH_N_ESTIMATORS = 200
FINAL_N_ESTIMATORS = 500
C_FN, C_FP = 420.0, 6.0


## 1. Load features + same train/cal/test split as notebook 02

Using the identical `random_state=42` split means the held-out test set here is the same one notebook 02 evaluated on — results are directly comparable.

In [ ]:
DROP_COLS = [
    "TransactionID", "TransactionDT", "card2", "card3", "card5", "addr1", "addr2",
    "P_emaildomain", "R_emaildomain", "DeviceInfo", "DeviceType", "card4", "card6", "ProductCD",
    "M1","M2","M3","M4","M5","M6","M7","M8","M9",
    "id_12","id_15","id_16","id_23","id_27","id_28",
    "id_29","id_30","id_31","id_32","id_33","id_34","id_35","id_36","id_37","id_38",
]

df_full = pd.read_parquet(DATA_DIR / "train_engineered.parquet")
y = df_full["isFraud"].astype(np.int8)
X = df_full.drop(columns=[c for c in DROP_COLS if c in df_full.columns] + ["isFraud"])
for col in X.select_dtypes("object").columns:
    X[col] = X[col].astype("category").cat.codes

X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
X_train, X_cal, y_train, y_cal = train_test_split(X_trainval, y_trainval, test_size=0.125, stratify=y_trainval, random_state=42)

neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
spw = neg / pos
print(f"Train: {len(X_train):,}  Cal: {len(X_cal):,}  Test: {len(X_test):,}  scale_pos_weight={spw:.1f}")


## 2. Stratified subsample for the search phase

25 trials × 2 methods × 3-fold CV = 150 XGBoost fits. Fitting each on the full 472K-row training set would make the search itself the bottleneck, so we search on a 60K-row stratified subsample (fixed `n_estimators=200`, no early stopping) and only train the *final* candidates at full scale.

In [ ]:
sub_idx = X_trainval.sample(n=min(SEARCH_SUBSAMPLE, len(X_trainval)), random_state=42).index
X_sub, y_sub = X_trainval.loc[sub_idx], y_trainval.loc[sub_idx]
sub_spw = (y_sub == 0).sum() / (y_sub == 1).sum()
folds = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42)
print(f"Search subsample: {len(X_sub):,} rows  |  {CV_FOLDS}-fold CV  |  {N_TRIALS} trials/method")


## 3. CV scoring function

Shared by both search methods so the comparison is apples-to-apples: same folds, same metric (PR-AUC), same fixed `n_estimators`.

In [ ]:
def cv_score(params, X, y, spw, folds):
    scores = []
    for train_idx, val_idx in folds.split(X, y):
        clf = XGBClassifier(
            n_estimators=SEARCH_N_ESTIMATORS, scale_pos_weight=spw, eval_metric="aucpr",
            random_state=42, n_jobs=-1, tree_method="hist", **params,
        )
        clf.fit(X.iloc[train_idx], y.iloc[train_idx])
        proba = clf.predict_proba(X.iloc[val_idx])[:, 1]
        scores.append(average_precision_score(y.iloc[val_idx], proba))
    return float(np.mean(scores))

def finalize_method(name, trials, best_so_far, total_time):
    final_best = best_so_far[-1]
    target = 0.95 * final_best
    trials_to_95 = next((i for i, s in enumerate(best_so_far) if s >= target), len(best_so_far) - 1)
    best_trial = max(trials, key=lambda t: t["cv_score"])
    return {
        "method": name, "trials": trials, "best_so_far": [round(s, 4) for s in best_so_far],
        "total_time_sec": round(total_time, 1), "best_cv_score": round(final_best, 4),
        "trials_to_95pct": trials_to_95, "best_params": best_trial["params"],
    }


## 4. RandomizedSearchCV — blind sampling

Each trial samples hyperparameters independently, with no knowledge of prior trials' results.

In [ ]:
def sample_random_params(rng):
    params = {}
    for name, spec in SEARCH_SPACE.items():
        if len(spec) == 3:
            lo, hi, _ = spec
            params[name] = float(np.exp(rng.uniform(np.log(lo), np.log(hi))))
        elif name in ("max_depth", "min_child_weight"):
            lo, hi = spec
            params[name] = int(rng.integers(lo, hi + 1))
        else:
            lo, hi = spec
            params[name] = float(rng.uniform(lo, hi))
    return params

rng = np.random.default_rng(42)
trials, best_so_far, best_score = [], [], -np.inf
t_start = time.time()
for i in range(N_TRIALS):
    params = sample_random_params(rng)
    t0 = time.time()
    score = cv_score(params, X_sub, y_sub, sub_spw, folds)
    best_score = max(best_score, score)
    best_so_far.append(best_score)
    trials.append({"trial": i, "params": params, "cv_score": round(score, 4), "fit_time_sec": round(time.time() - t0, 2)})
random_result = finalize_method("random_search", trials, best_so_far, time.time() - t_start)
print(f"RandomizedSearch: best_cv={random_result['best_cv_score']:.4f}  time={random_result['total_time_sec']:.1f}s  trials_to_95%={random_result['trials_to_95pct']}")


## 5. Optuna — Bayesian (TPE) sampling

Each trial's proposal is informed by the posterior over past trials' scores — it should need fewer trials to reach the same quality, since it stops wasting fits on regions that already look bad.

In [ ]:
def objective(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", *SEARCH_SPACE["max_depth"]),
        "learning_rate": trial.suggest_float("learning_rate", *SEARCH_SPACE["learning_rate"][:2], log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", *SEARCH_SPACE["min_child_weight"]),
        "subsample": trial.suggest_float("subsample", *SEARCH_SPACE["subsample"]),
        "colsample_bytree": trial.suggest_float("colsample_bytree", *SEARCH_SPACE["colsample_bytree"]),
        "gamma": trial.suggest_float("gamma", *SEARCH_SPACE["gamma"]),
        "reg_alpha": trial.suggest_float("reg_alpha", *SEARCH_SPACE["reg_alpha"][:2], log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", *SEARCH_SPACE["reg_lambda"][:2], log=True),
    }
    t0 = time.time()
    score = cv_score(params, X_sub, y_sub, sub_spw, folds)
    trial_times.append(time.time() - t0)
    return score

trial_times = []
t_start = time.time()
study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)
total_time = time.time() - t_start

trials, best_so_far, best_score = [], [], -np.inf
for i, t in enumerate(study.trials):
    best_score = max(best_score, t.value)
    best_so_far.append(best_score)
    trials.append({"trial": i, "params": {k: (float(v) if isinstance(v, float) else int(v)) for k, v in t.params.items()},
                    "cv_score": round(t.value, 4), "fit_time_sec": round(trial_times[i], 2)})
optuna_result = finalize_method("optuna", trials, best_so_far, total_time)
print(f"Optuna: best_cv={optuna_result['best_cv_score']:.4f}  time={optuna_result['total_time_sec']:.1f}s  trials_to_95%={optuna_result['trials_to_95pct']}")


## 6. Verdict — which search method wins, and why

Win on **both** axes → clear call. Split result (one faster, one better) → prefer final quality, since the trial budget here is fixed and small; sample efficiency matters most when trials are cheap or the budget is open-ended.

In [ ]:
faster = min(random_result, optuna_result, key=lambda r: r["trials_to_95pct"])
higher = max(random_result, optuna_result, key=lambda r: r["best_cv_score"])
if faster["method"] == higher["method"]:
    winner = faster["method"]
    reason = f"{winner} won on both cost and performance — higher CV PR-AUC in fewer trials."
else:
    winner = higher["method"]
    reason = (f"{higher['method']} found the better final config; {faster['method']} converged faster. "
              f"Each fit here is expensive, so sample efficiency matters, but final quality decides "
              f"under a fixed trial budget.")
print(f"Winner: {winner}\n{reason}")


## 7. Final full-scale fit + evaluation

Retrain three candidates at full scale (472K rows, 500 estimators, early stopping, isotonic calibration) and compare on the *same* held-out test set used in notebook 02: the manual baseline, RandomizedSearch's best config, and Optuna's best config.

In [ ]:
def fit_and_eval(params, X_train, y_train, X_cal, y_cal, X_test, y_test, spw):
    clf = XGBClassifier(
        n_estimators=FINAL_N_ESTIMATORS, scale_pos_weight=spw, eval_metric="aucpr",
        early_stopping_rounds=30, random_state=42, n_jobs=-1, tree_method="hist", **params,
    )
    clf.fit(X_train, y_train, eval_set=[(X_cal, y_cal)], verbose=False)

    raw_cal = clf.predict_proba(X_cal)[:, 1]
    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(raw_cal, y_cal)

    y_prob = iso.predict(clf.predict_proba(X_test)[:, 1])
    pr_auc = average_precision_score(y_test, y_prob)
    y_pred = (y_prob >= 0.5).astype(int)
    fb2 = fbeta_score(y_test, y_pred, beta=2, zero_division=0)
    fn = int(((y_test == 1) & (y_pred == 0)).sum())
    fp = int(((y_test == 0) & (y_pred == 1)).sum())
    cost_per_day = (C_FN * fn + C_FP * fp) / 30.0
    return {"pr_auc": round(float(pr_auc), 4), "fbeta2": round(float(fb2), 4),
            "cost_per_day": round(float(cost_per_day), 0), "params": params, "model": clf, "iso": iso}

final_results = {}
for label, params in [("baseline_manual", BASELINE_PARAMS),
                       ("random_search_tuned", random_result["best_params"]),
                       ("optuna_tuned", optuna_result["best_params"])]:
    final_results[label] = fit_and_eval(params, X_train, y_train, X_cal, y_cal, X_test, y_test, spw)
    r = final_results[label]
    print(f"{label:22s}  PR-AUC={r['pr_auc']:.4f}  F-beta2={r['fbeta2']:.4f}  cost/day=${r['cost_per_day']:,.0f}")


## 8. Save tuned model + export results

In [ ]:
import pickle

tuned_label = "random_search_tuned" if final_results["random_search_tuned"]["pr_auc"] >= final_results["optuna_tuned"]["pr_auc"] else "optuna_tuned"
tuned = final_results[tuned_label]
model_path = MODEL_DIR / "fraud_xgb_tuned.pkl"
with open(model_path, "wb") as f:
    pickle.dump({"iso": tuned["iso"], "xgb_base": tuned["model"], "feature_names": list(X.columns)}, f)
print(f"Tuned model ({tuned_label}) saved -> {model_path}")

with open(EXPORT_PATH) as f:
    export = json.load(f)

export["hyperparam_search"] = {
    "search_space": {k: list(v) for k, v in SEARCH_SPACE.items()},
    "trial_budget": N_TRIALS, "cv_folds": CV_FOLDS, "subsample_size": len(X_sub),
    "methods": {
        "random_search": {k: v for k, v in random_result.items() if k != "trials"} | {
            "trials": [{"trial": t["trial"], "cv_score": t["cv_score"], "fit_time_sec": t["fit_time_sec"]} for t in random_result["trials"]]},
        "optuna": {k: v for k, v in optuna_result.items() if k != "trials"} | {
            "trials": [{"trial": t["trial"], "cv_score": t["cv_score"], "fit_time_sec": t["fit_time_sec"]} for t in optuna_result["trials"]]},
    },
    "winner": winner, "verdict_reason": reason,
    "final_eval": {label: {k: v for k, v in res.items() if k not in ("model", "iso")} for label, res in final_results.items()},
    "tuned_model": tuned_label,
}
with open(EXPORT_PATH, "w") as f:
    json.dump(export, f)
print(f"Saved -> {EXPORT_PATH}")
